[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/100.%20The%20Closed-Loop%20%28Reverse%20Logistics%29%20Network%20Design%20Problem/P100-Tier-5_executed.ipynb)

---


# 100. The Closed-Loop Network Design Problem
## Tier 5: An Integrated Digital Twin Simulation

### Purpose and scope

This notebook builds a small, illustrative digital-twin simulation for a logistics network. It represents sensors, physical entities and their virtual counterparts, predictive models, operational metrics, and what-if scenarios. The code then advances the simulated system, records its state, and visualizes selected indicators.

The notebook is an educational simulation, not a deployed digital twin or a network-design optimizer. It does not connect to physical devices, receive live IoT streams, solve a facility-location or routing model, or validate predictions against observed outcomes. Sensor readings, model predictions, synchronization behavior, costs, and scenario impacts are generated by simplified rules. As a result, the reported metrics illustrate how components can be connected; they are not operational forecasts or performance guarantees.

### Inputs and outputs

- **Inputs:** fixed sensor, entity, and model configurations; seeded random-number generators; and scenario definitions.
- **Process:** generate sensor readings, synchronize virtual entity states, run simple predictive models, update metrics, and simulate selected scenarios.
- **Outputs:** a time series of simulated readings and metrics, a system report, scenario summaries, and plots.

### Modeling assumptions

- The example has seven sensor types with eight sensors of each type, for 56 sensors in total.
- Sensor values are synthetic and use type-specific scales. Readings of different types should not be compared as though they shared a physical unit.
- Each digital entity stores a physical state and a virtual state. Synchronization quality is a simplified indicator that decays with elapsed time since synchronization.
- Predictive models use simple stochastic rules. Their confidence values are illustrative, not measured accuracy.
- Scenario impacts are prescribed multipliers, not outputs from an optimization model.
- Simulation timestamps use the computer's clock; the loop includes a small delay but does not guarantee real-time timing or sub-second latency.

### Solution workflow

1. Define the sensor, entity, predictive-model, and digital-twin data structures.
2. Instantiate the example system and its operational metrics.
3. Repeatedly update sensors, synchronize entities, run predictions, and evaluate scenarios.
4. Record the resulting time series and generate a report.
5. Visualize synchronization, sensor readings, prediction confidence, and selected operational metrics.

This workflow is a component-level digital-twin demonstration. It does not implement the closed-loop network-design optimization problem itself.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass, field
from enum import Enum
import random
import time
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixed seeds make the synthetic sensor and model draws reproducible.
np.random.seed(42)
random.seed(42)

In [ ]:
class SensorType(Enum):
    """Sensor categories represented in the example logistics system."""

    TEMPERATURE = "temperature"
    HUMIDITY = "humidity"
    MOTION = "motion"
    INVENTORY = "inventory"
    TRAFFIC = "traffic"
    ENERGY = "energy"
    QUALITY = "quality"


@dataclass
class IoTSensor:
    """A sensor record with a location, reading, and operational status."""

    sensor_id: str
    sensor_type: SensorType
    location: Tuple[float, float]
    value: float = 0.0
    last_update: float = 0.0
    accuracy: float = 0.95
    status: str = "active"

    def update_value(self, new_value: float, timestamp: float) -> None:
        """Store a reading and timestamp, with small simulated measurement noise."""
        self.value = new_value + np.random.normal(0, 0.02)
        self.last_update = timestamp


@dataclass
class DigitalEntity:
    """A virtual record paired with a physical logistics entity."""

    entity_id: str
    entity_type: str
    location: Tuple[float, float]
    physical_state: Dict[str, Any] = field(default_factory=dict)
    virtual_state: Dict[str, Any] = field(default_factory=dict)
    last_sync: float = 0.0
    sync_quality: float = 1.0

    def synchronize(self, timestamp: float) -> float:
        """Refresh the virtual state and return a simple freshness score.

        The score is 1 for a repeated timestamp and otherwise decreases with
        elapsed time, with a floor of 0.5. Numeric state values receive a small
        simulated divergence; nonnumeric values, such as status labels, are
        copied without arithmetic.
        """
        time_diff = max(0.0, timestamp - self.last_sync)
        if time_diff == 0:
            self.sync_quality = 1.0
        else:
            self.sync_quality = max(0.5, 1.0 - time_diff / 3600)

        self.last_sync = timestamp

        for key, physical_value in self.physical_state.items():
            if isinstance(physical_value, (int, float)) and not isinstance(physical_value, bool):
                # Less-fresh synchronization permits more simulated state divergence.
                divergence = np.random.normal(0, 0.05) * (1.0 - self.sync_quality)
                self.virtual_state[key] = physical_value * (1.0 + divergence)
            else:
                self.virtual_state[key] = physical_value

        return self.sync_quality


@dataclass
class PredictiveModel:
    """A lightweight stochastic stand-in for a trained predictive model."""

    model_id: str
    model_type: str
    accuracy: float = 0.8
    training_data: List[Dict[str, Any]] = field(default_factory=list)
    predictions: List[Dict[str, Any]] = field(default_factory=list)

    def train(self, data: List[Dict[str, Any]]) -> None:
        """Store examples and simulate a small accuracy increase after 100 records."""
        self.training_data.extend(data)
        if len(self.training_data) > 100:
            self.accuracy = min(0.95, self.accuracy + 0.01)

    def predict(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        """Return a synthetic prediction for the model's declared task."""
        if self.model_type == "demand_forecast":
            base_demand = input_data.get("current_demand", 100)
            predicted_demand = base_demand * np.random.uniform(0.8, 1.2)
            confidence = self.accuracy * np.random.uniform(0.8, 1.0)
            return {
                "predicted_demand": predicted_demand,
                "confidence": confidence,
                "timestamp": time.time(),
            }

        if self.model_type == "route_optimization":
            current_cost = input_data.get("current_cost", 1000)
            optimized_cost = current_cost * np.random.uniform(0.7, 0.95)
            return {
                "optimized_cost": optimized_cost,
                "savings": current_cost - optimized_cost,
                "timestamp": time.time(),
            }

        return {
            "prediction": np.random.uniform(0, 1),
            "confidence": self.accuracy,
            "timestamp": time.time(),
        }


@dataclass
class DigitalTwin:
    """Container coordinating sensors, entities, models, and metrics."""

    twin_id: str
    sensors: List[IoTSensor] = field(default_factory=list)
    entities: List[DigitalEntity] = field(default_factory=list)
    predictive_models: List[PredictiveModel] = field(default_factory=list)
    synchronization_status: float = 1.0
    prediction_accuracy: float = 0.8
    operational_metrics: Dict[str, float] = field(default_factory=dict)

    def add_sensor(self, sensor: IoTSensor) -> None:
        self.sensors.append(sensor)

    def add_entity(self, entity: DigitalEntity) -> None:
        self.entities.append(entity)

    def add_predictive_model(self, model: PredictiveModel) -> None:
        self.predictive_models.append(model)

    def update_sensors(self, timestamp: float) -> None:
        """Generate one synthetic reading for each active sensor."""
        for sensor in self.sensors:
            if sensor.status != "active":
                continue

            if sensor.sensor_type == SensorType.TEMPERATURE:
                base_value = 20 + 5 * np.sin(timestamp / 3600 * 2 * np.pi)
                new_value = base_value + np.random.normal(0, 2)
            elif sensor.sensor_type == SensorType.HUMIDITY:
                base_value = 50 + 10 * np.sin(timestamp / 3600 * 2 * np.pi + np.pi / 4)
                new_value = base_value + np.random.normal(0, 5)
            elif sensor.sensor_type == SensorType.MOTION:
                new_value = np.random.uniform(0, 100)
            elif sensor.sensor_type == SensorType.INVENTORY:
                new_value = np.random.uniform(0, 1000)
            elif sensor.sensor_type == SensorType.TRAFFIC:
                new_value = np.random.uniform(0, 50)
            elif sensor.sensor_type == SensorType.ENERGY:
                new_value = np.random.uniform(100, 500)
            elif sensor.sensor_type == SensorType.QUALITY:
                new_value = np.random.uniform(0.8, 1.0)
            else:
                new_value = np.random.uniform(0, 100)

            sensor.update_value(new_value, timestamp)

    def synchronize_entities(self, timestamp: float) -> float:
        """Synchronize every entity and average their freshness scores."""
        sync_qualities = [entity.synchronize(timestamp) for entity in self.entities]
        self.synchronization_status = (
            float(np.mean(sync_qualities)) if sync_qualities else 1.0
        )
        return self.synchronization_status

    def run_predictions(self) -> List[Dict[str, Any]]:
        """Run each model using current sensor readings and operational metrics."""
        active_sensors = [sensor for sensor in self.sensors if sensor.status == "active"]
        input_data = {
            sensor_type.value: float(
                np.mean([s.value for s in active_sensors if s.sensor_type == sensor_type])
            )
            for sensor_type in SensorType
            if any(s.sensor_type == sensor_type for s in active_sensors)
        }
        if "inventory" in input_data:
            input_data["current_demand"] = input_data["inventory"]
        input_data["current_cost"] = self.operational_metrics.get("total_cost", 1000)

        predictions = []
        for model in self.predictive_models:
            prediction = model.predict(input_data)
            prediction["model_id"] = model.model_id
            prediction["model_type"] = model.model_type
            model.predictions.append(prediction)
            predictions.append(prediction)
        return predictions

    def simulate_scenario(self, scenario_config: Dict[str, Any]) -> Dict[str, Any]:
        """Apply prescribed scenario multipliers to current cost and service."""
        scenario_type = scenario_config.get("type", "demand_increase")
        scenario_magnitude = scenario_config.get("magnitude", 1.2)

        if scenario_type == "demand_increase":
            impact_on_cost = scenario_magnitude * 1.15
            impact_on_service = 1.0 / scenario_magnitude
        elif scenario_type == "facility_disruption":
            impact_on_cost = 1.5
            impact_on_service = 0.7
        elif scenario_type == "route_optimization":
            impact_on_cost = 0.85
            impact_on_service = 1.05
        else:
            impact_on_cost = 1.0
            impact_on_service = 1.0

        base_cost = self.operational_metrics.get("total_cost", 10000)
        base_service = self.operational_metrics.get("service_level", 0.95)
        return {
            "scenario_type": scenario_type,
            "scenario_magnitude": scenario_magnitude,
            "predicted_cost": base_cost * impact_on_cost,
            "predicted_service_level": base_service * impact_on_service,
            "cost_impact": (impact_on_cost - 1.0) * 100,
            "service_impact": (impact_on_service - 1.0) * 100,
            "recommendation": self._generate_scenario_recommendation(
                scenario_type, impact_on_cost, impact_on_service
            ),
        }

    def _generate_scenario_recommendation(
        self, scenario_type: str, cost_impact: float, service_impact: float
    ) -> str:
        """Translate scenario multipliers into a simple illustrative message."""
        if cost_impact > 1.2:
            return "High cost impact: consider capacity expansion or cost optimization."
        if service_impact < 0.8:
            return "Service degradation: consider service recovery measures."
        if cost_impact < 0.9 and service_impact > 1.0:
            return "Positive scenario: consider scaling successful strategies."
        return "Monitor the situation and prepare contingency plans."

    def update_operational_metrics(self) -> None:
        """Summarize active sensor readings and record illustrative system metrics."""
        active_sensors = [sensor for sensor in self.sensors if sensor.status == "active"]

        def average_reading(sensor_type: SensorType, default: float) -> float:
            values = [s.value for s in active_sensors if s.sensor_type == sensor_type]
            return float(np.mean(values)) if values else default

        total_energy = sum(
            sensor.value
            for sensor in active_sensors
            if sensor.sensor_type == SensorType.ENERGY
        )
        self.operational_metrics = {
            "total_cost": float(np.random.uniform(8000, 12000)),
            "service_level": average_reading(SensorType.QUALITY, 0.9),
            "energy_consumption": float(total_energy),
            "temperature": average_reading(SensorType.TEMPERATURE, 20.0),
            "humidity": average_reading(SensorType.HUMIDITY, 50.0),
            "sensor_coverage": len(active_sensors) / len(self.sensors) if self.sensors else 0.0,
            "entity_sync_quality": self.synchronization_status,
            "prediction_accuracy": self.prediction_accuracy,
        }

    def generate_twin_report(self) -> Dict[str, Any]:
        """Summarize system composition, metrics, and simple threshold-based advice."""
        active_sensors = [sensor for sensor in self.sensors if sensor.status == "active"]
        report = {
            "report_timestamp": time.time(),
            "twin_id": self.twin_id,
            "system_overview": {
                "total_sensors": len(self.sensors),
                "active_sensors": len(active_sensors),
                "total_entities": len(self.entities),
                "total_models": len(self.predictive_models),
                "synchronization_status": self.synchronization_status,
                "prediction_accuracy": self.prediction_accuracy,
            },
            "sensor_analysis": {},
            "entity_analysis": {},
            "model_analysis": {},
            "operational_metrics": self.operational_metrics.copy(),
            "recommendations": [],
        }

        sensor_types = defaultdict(int)
        for sensor in self.sensors:
            sensor_types[sensor.sensor_type.value] += 1
        report["sensor_analysis"] = {
            "sensor_types": dict(sensor_types),
            "average_accuracy": float(np.mean([s.accuracy for s in self.sensors]))
            if self.sensors else 0.0,
            "coverage_area": len(self.sensors) * 10,
        }

        entity_types = defaultdict(int)
        for entity in self.entities:
            entity_types[entity.entity_type] += 1
        report["entity_analysis"] = {
            "entity_types": dict(entity_types),
            "average_sync_quality": float(np.mean([e.sync_quality for e in self.entities]))
            if self.entities else 0.0,
            "total_entities": len(self.entities),
        }

        model_types = defaultdict(int)
        for model in self.predictive_models:
            model_types[model.model_type] += 1
        report["model_analysis"] = {
            "model_types": dict(model_types),
            "average_accuracy": float(np.mean([m.accuracy for m in self.predictive_models]))
            if self.predictive_models else 0.0,
            "total_predictions": sum(len(model.predictions) for model in self.predictive_models),
        }

        if self.synchronization_status < 0.8:
            report["recommendations"].append("Review synchronization frequency and data quality.")
        if len(active_sensors) < len(self.sensors) * 0.8:
            report["recommendations"].append("Review sensor health and maintenance status.")
        if self.prediction_accuracy < 0.8:
            report["recommendations"].append("Review predictive-model data and evaluation.")
        if self.operational_metrics.get("service_level", 0) < 0.9:
            report["recommendations"].append("Review the factors affecting the service-level metric.")
        return report

In [ ]:
print("Initializing the integrated digital-twin example...")
print("=" * 52)

digital_twin = DigitalTwin(twin_id="closed_loop_twin_001")

sensor_types = [
    SensorType.TEMPERATURE,
    SensorType.HUMIDITY,
    SensorType.MOTION,
    SensorType.INVENTORY,
    SensorType.TRAFFIC,
    SensorType.ENERGY,
    SensorType.QUALITY,
]
num_sensors_per_type = 8
print(f"Creating {len(sensor_types) * num_sensors_per_type} synthetic sensors...")

for sensor_type in sensor_types:
    for index in range(num_sensors_per_type):
        digital_twin.add_sensor(
            IoTSensor(
                sensor_id=f"{sensor_type.value}_{index}",
                sensor_type=sensor_type,
                location=(np.random.uniform(0, 100), np.random.uniform(0, 100)),
                accuracy=np.random.uniform(0.9, 0.99),
            )
        )

print(f"Sensors created: {len(digital_twin.sensors)}")
for sensor_type in sensor_types:
    count = sum(sensor.sensor_type == sensor_type for sensor in digital_twin.sensors)
    print(f"  {sensor_type.value.title()}: {count}")

entity_configurations = [
    ("facility_1", "facility", (20, 30)),
    ("facility_2", "facility", (60, 70)),
    ("facility_3", "facility", (40, 50)),
    ("vehicle_1", "vehicle", (25, 35)),
    ("vehicle_2", "vehicle", (45, 55)),
    ("vehicle_3", "vehicle", (65, 75)),
    ("product_a", "product", (30, 40)),
    ("product_b", "product", (50, 60)),
    ("customer_1", "customer", (15, 25)),
    ("customer_2", "customer", (55, 65)),
]
print("\nCreating digital entities...")

for entity_id, entity_type, location in entity_configurations:
    digital_twin.add_entity(
        DigitalEntity(
            entity_id=entity_id,
            entity_type=entity_type,
            location=location,
            physical_state={
                "status": "active",
                "capacity": np.random.uniform(100, 1000),
                "utilization": np.random.uniform(0.3, 0.9),
                "efficiency": np.random.uniform(0.7, 0.95),
            },
        )
    )

print(f"Digital entities created: {len(digital_twin.entities)}")
for entity_type in ["facility", "vehicle", "product", "customer"]:
    count = sum(entity.entity_type == entity_type for entity in digital_twin.entities)
    print(f"  {entity_type.title()}: {count}")

model_configurations = [
    ("demand_forecast_001", "demand_forecast"),
    ("route_optimization_001", "route_optimization"),
    ("maintenance_prediction_001", "maintenance_prediction"),
]
print("\nCreating illustrative predictive models...")

for model_id, model_type in model_configurations:
    digital_twin.add_predictive_model(
        PredictiveModel(
            model_id=model_id,
            model_type=model_type,
            accuracy=np.random.uniform(0.75, 0.9),
        )
    )

print(f"Predictive models created: {len(digital_twin.predictive_models)}")
for model in digital_twin.predictive_models:
    print(f"  {model.model_id}: {model.model_type.replace('_', ' ').title()}")

digital_twin.update_operational_metrics()
print("\nDigital-twin example initialized.")
print(f"  Twin ID: {digital_twin.twin_id}")
print(f"  Sensors: {len(digital_twin.sensors)}")
print(f"  Entities: {len(digital_twin.entities)}")
print(f"  Predictive models: {len(digital_twin.predictive_models)}")

## Simulation logic

At each simulated update, the system records the current time, generates readings for active sensors, synchronizes entity states, refreshes operational metrics, and stores the resulting values. Predictions are run every 10 loop iterations and scenarios are evaluated every 30 loop iterations. These intervals are counts of iterations, not guarantees about wall-clock seconds.

The simulation returns a dictionary of time series. For example, `simulation_data["synchronization_status"][i]` is the synchronization score recorded at the same update as `simulation_data["timestamps"][i]`. Keeping these values in parallel lists makes it possible to plot and analyze the evolution of the simulated system.

In [ ]:
def run_digital_twin_simulation(
    digital_twin: DigitalTwin, duration_minutes: int = 5
) -> Dict[str, List[Any]]:
    """Run a short, iteration-based simulation and collect its time series."""
    if duration_minutes < 0:
        raise ValueError("duration_minutes must be nonnegative")

    print(f"Running the digital-twin simulation for {duration_minutes} minutes...")
    print("=" * 60)
    simulation_data: Dict[str, List[Any]] = {
        "timestamps": [],
        "synchronization_status": [],
        "sensor_readings": [],
        "predictions": [],
        "operational_metrics": [],
        "scenario_simulations": [],
    }
    start_time = time.time()

    for minute in range(duration_minutes):
        for second in range(60):
            current_time = time.time()
            digital_twin.update_sensors(current_time)
            sync_status = digital_twin.synchronize_entities(current_time)

            if second % 10 == 0:
                predictions = digital_twin.run_predictions()
                training_record = {
                    sensor.sensor_type.value: sensor.value
                    for sensor in digital_twin.sensors
                    if sensor.status == "active"
                }
                for model in digital_twin.predictive_models:
                    model.train([training_record.copy()])
            else:
                predictions = []

            digital_twin.update_operational_metrics()

            if second % 30 == 0:
                scenarios = [
                    {"type": "demand_increase", "magnitude": 1.2},
                    {"type": "facility_disruption", "magnitude": 1.0},
                    {"type": "route_optimization", "magnitude": 1.0},
                ]
                scenario_result = digital_twin.simulate_scenario(random.choice(scenarios))
            else:
                scenario_result = None

            simulation_data["timestamps"].append(current_time)
            simulation_data["synchronization_status"].append(sync_status)
            simulation_data["sensor_readings"].append(
                {
                    sensor.sensor_id: sensor.value
                    for sensor in digital_twin.sensors
                    if sensor.status == "active"
                }
            )
            simulation_data["predictions"].append(predictions)
            simulation_data["operational_metrics"].append(
                digital_twin.operational_metrics.copy()
            )
            simulation_data["scenario_simulations"].append(scenario_result)

            if second == 0:
                active_count = sum(
                    sensor.status == "active" for sensor in digital_twin.sensors
                )
                print(
                    f"Minute {minute + 1}: synchronization={sync_status:.3f}, "
                    f"active sensors={active_count}, entities={len(digital_twin.entities)}"
                )
                for prediction in predictions:
                    if prediction["model_type"] == "demand_forecast":
                        print(
                            f"  Demand forecast: {prediction['predicted_demand']:.1f}; "
                            f"illustrative confidence={prediction['confidence']:.2f}"
                        )
                    elif prediction["model_type"] == "route_optimization":
                        print(
                            f"  Synthetic route cost: ${prediction['optimized_cost']:.0f}; "
                            f"synthetic savings=${prediction['savings']:.0f}"
                        )
                if scenario_result is not None:
                    print(
                        f"  Scenario: {scenario_result['scenario_type'].replace('_', ' ')}, "
                        f"cost impact={scenario_result['cost_impact']:+.1f}%, "
                        f"service impact={scenario_result['service_impact']:+.1f}%"
                    )

            time.sleep(0.01)

    elapsed_time = time.time() - start_time
    update_count = len(simulation_data["timestamps"])
    prediction_count = sum(len(items) for items in simulation_data["predictions"])
    scenario_count = sum(
        scenario is not None for scenario in simulation_data["scenario_simulations"]
    )
    print("\nSimulation summary:")
    print(f"  Wall-clock runtime: {elapsed_time:.2f} seconds")
    print(f"  Recorded updates: {update_count}")
    if update_count:
        print(
            "  Mean synchronization score: "
            f"{np.mean(simulation_data['synchronization_status']):.3f}"
        )
    print(f"  Recorded predictions: {prediction_count}")
    print(f"  Scenario evaluations: {scenario_count}")
    return simulation_data


simulation_data = run_digital_twin_simulation(digital_twin, duration_minutes=3)
final_report = digital_twin.generate_twin_report()

print("\nFinal digital-twin report")
print("=" * 30)
overview = final_report["system_overview"]
print(f"Total sensors: {overview['total_sensors']}")
print(f"Active sensors: {overview['active_sensors']}")
print(f"Total entities: {overview['total_entities']}")
print(f"Total predictive models: {overview['total_models']}")
print(f"Mean entity synchronization score: {overview['synchronization_status']:.3f}")

print("\nOperational metrics:")
for metric, value in final_report["operational_metrics"].items():
    print(f"  {metric.replace('_', ' ').title()}: {value:.3f}")

print("\nRule-based recommendations:")
if final_report["recommendations"]:
    for index, recommendation in enumerate(final_report["recommendations"], start=1):
        print(f"  {index}. {recommendation}")
else:
    print("  No recommendation thresholds were triggered.")

## Visualizing and interpreting the simulation

The plots summarize different aspects of the recorded run. Synchronization scores describe the model's freshness indicator; the sensor heatmap shows readings in each sensor type's own scale; and the demand plot shows the synthetic confidence values returned by the demand-forecast model. The service and energy plot uses two vertical axes because the quantities have different scales. Energy is the sum of the active energy-sensor readings as represented by this example, not a calibrated measurement.

These plots are descriptive. In particular, a confidence value generated by the example model is not evidence of predictive accuracy, and a scenario result reflects its prescribed multipliers rather than a validated operational forecast.

In [ ]:
def visualize_digital_twin_performance(
    digital_twin: DigitalTwin, simulation_data: Dict[str, List[Any]]
) -> Dict[str, List[Any]]:
    """Plot recorded indicators and print descriptive summaries."""
    timestamps = simulation_data["timestamps"]
    if not timestamps:
        raise ValueError("The simulation data contain no recorded updates.")

    time_minutes = [(timestamp - timestamps[0]) / 60 for timestamp in timestamps]
    synchronization = simulation_data["synchronization_status"]
    if len(time_minutes) != len(synchronization):
        raise ValueError("Timestamps and synchronization scores must have equal lengths.")

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

    ax1.set_title("Digital-Twin Synchronization Score")
    ax1.plot(time_minutes, synchronization, color="blue", linewidth=2)
    ax1.axhline(0.8, color="red", linestyle="--", alpha=0.7, label="Reference: 0.8")
    ax1.axhline(0.9, color="green", linestyle="--", alpha=0.7, label="Reference: 0.9")
    ax1.set_xlabel("Elapsed time (minutes)")
    ax1.set_ylabel("Synchronization score (0–1)")
    ax1.set_ylim(0, 1)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.set_title("Sensor Readings by Type")
    active_sensors = [sensor for sensor in digital_twin.sensors if sensor.status == "active"]
    sensor_matrix = []
    sensor_labels = []
    for sensor_type in SensorType:
        type_sensors = [sensor for sensor in active_sensors if sensor.sensor_type == sensor_type]
        if not type_sensors:
            continue
        readings = []
        for reading_data in simulation_data["sensor_readings"]:
            values = [reading_data[s.sensor_id] for s in type_sensors if s.sensor_id in reading_data]
            readings.append(float(np.mean(values)) if values else np.nan)
        sensor_matrix.append(readings)
        sensor_labels.append(sensor_type.value.title())

    if sensor_matrix:
        sample_count = min(50, len(time_minutes))
        sample_indices = np.linspace(0, len(time_minutes) - 1, sample_count, dtype=int)
        heatmap_data = np.asarray(sensor_matrix)[:, sample_indices]
        image = ax2.imshow(heatmap_data, cmap="viridis", aspect="auto")
        ax2.set_yticks(range(len(sensor_labels)))
        ax2.set_yticklabels(sensor_labels)
        tick_positions = np.linspace(0, sample_count - 1, min(5, sample_count), dtype=int)
        ax2.set_xticks(tick_positions)
        ax2.set_xticklabels([f"{time_minutes[sample_indices[i]]:.2f}" for i in tick_positions])
        ax2.set_xlabel("Elapsed time (minutes; sampled)")
        colorbar = fig.colorbar(image, ax=ax2)
        colorbar.set_label("Reading (sensor-type-specific scale)")
    else:
        ax2.text(0.5, 0.5, "No active sensor readings", ha="center", va="center")
        ax2.set_axis_off()

    ax3.set_title("Synthetic Demand-Forecast Confidence")
    prediction_times = []
    demand_confidences = []
    for index, predictions in enumerate(simulation_data["predictions"]):
        for prediction in predictions:
            if prediction["model_type"] == "demand_forecast":
                prediction_times.append(time_minutes[index])
                demand_confidences.append(prediction["confidence"])
    if demand_confidences:
        ax3.plot(
            prediction_times,
            demand_confidences,
            color="green",
            linewidth=1.5,
            marker="o",
            markersize=3,
            label="Demand forecast",
        )
        ax3.legend()
    else:
        ax3.text(0.5, 0.5, "No demand predictions recorded", ha="center", va="center")
    ax3.set_xlabel("Elapsed time (minutes)")
    ax3.set_ylabel("Illustrative confidence (0–1)")
    ax3.set_ylim(0, 1)
    ax3.grid(True, alpha=0.3)

    ax4.set_title("Selected Operational Metrics")
    metrics = simulation_data["operational_metrics"]
    service_levels = [item.get("service_level", 0) for item in metrics]
    energy_readings = [item.get("energy_consumption", 0) for item in metrics]
    ax4_twin = ax4.twinx()
    service_line, = ax4.plot(
        time_minutes, service_levels, color="blue", label="Service-level indicator", linewidth=2
    )
    energy_line, = ax4_twin.plot(
        time_minutes, energy_readings, color="red", label="Summed energy-sensor reading", linewidth=2
    )
    ax4.set_xlabel("Elapsed time (minutes)")
    ax4.set_ylabel("Service-level indicator (0–1)", color="blue")
    ax4.tick_params(axis="y", labelcolor="blue")
    ax4.set_ylim(0, 1)
    ax4_twin.set_ylabel("Summed energy-sensor reading (illustrative units)", color="red")
    ax4_twin.tick_params(axis="y", labelcolor="red")
    ax4.legend([service_line, energy_line], [service_line.get_label(), energy_line.get_label()], loc="upper left")
    ax4.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

    mean_sync = float(np.mean(synchronization))
    mean_service = float(np.mean(service_levels)) if service_levels else 0.0
    prediction_count = sum(len(items) for items in simulation_data["predictions"])
    active_count = len(active_sensors)
    coverage = active_count / len(digital_twin.sensors) if digital_twin.sensors else 0.0

    print("\nDescriptive simulation summary")
    print("=" * 36)
    print(f"Mean synchronization score: {mean_sync:.3f}")
    print(f"Mean service-level indicator: {mean_service:.3f}")
    print(f"Active sensors: {active_count}/{len(digital_twin.sensors)} ({coverage:.1%})")
    print(f"Recorded predictions: {prediction_count}")
    print(f"Demand predictions: {len(demand_confidences)}")
    if demand_confidences:
        print(f"Mean synthetic demand confidence: {np.mean(demand_confidences):.3f}")

    scenarios = [item for item in simulation_data["scenario_simulations"] if item is not None]
    if scenarios:
        scenario_counts = defaultdict(int)
        for scenario in scenarios:
            scenario_counts[scenario["scenario_type"]] += 1
        print(f"Scenario evaluations: {len(scenarios)}")
        for scenario_type, count in sorted(scenario_counts.items()):
            print(f"  {scenario_type.replace('_', ' ').title()}: {count}")

    return simulation_data


visualize_digital_twin_performance(digital_twin, simulation_data)

## Strengths, limitations, and appropriate use

### What this example demonstrates

- How sensor, entity, model, and metric records can be organized within one coordinating object.
- How repeated updates can create a time series for monitoring and analysis.
- How simple scenario multipliers can support what-if comparisons.
- How plots and a report can make selected simulated states easier to inspect.

### What it does not establish

- **Physical fidelity:** Synthetic readings and simplified state updates do not establish that the virtual system accurately represents a real operation.
- **Predictive validity:** The example's confidence values are generated by random rules; they are not calculated from held-out observations or a calibration procedure.
- **Optimization quality:** The route-cost calculation and scenario rules do not solve a network-design optimization problem.
- **Real-time performance:** A short sleep in a loop is not a timing guarantee, and the notebook does not implement a live synchronization service.
- **Operational readiness:** Production use would require, among other things, validated data interfaces, explicit units, fault handling, security controls, and domain-specific evaluation.

### Interpreting the results

A high synchronization score in this simulation means that the simplified freshness rule assigns a high score at the recorded update; it does not measure the fidelity of a real digital twin. Sensor coverage is the fraction of configured sensors marked active. Scenario cost and service impacts are consequences of the supplied multipliers. These results are useful for understanding the data flow and reporting structure, but should not be treated as evidence that an operational intervention will have the same effect.

### Suitable context

This tier is useful as a conceptual starting point when studying integrated monitoring, digital representations of physical entities, scenario analysis, and the data flow between these components. Applying the ideas to an actual closed-loop supply-chain network requires a validated network model, operational data, and decision logic appropriate to that application.